# PDF to Word Studio on Colab

Notebook nay dung cho Google Colab de:

1. Clone hoac pull repository tu GitHub.
2. Cai `MinerU` va dependencies cho Flask webapp PDF -> Word.
3. Chay webapp tren Colab va kiem tra `/api/status`.
4. Mo Cloudflare tunnel de lay URL public.
5. Backup ket qua chuyen doi len Google Drive neu can.

Mac dinh notebook chay backend `auto`:
- Neu co `MINERU_API_URL`: dung `vlm-http-client`.
- Neu runtime co CUDA: dung `hybrid-auto-engine`.
- Neu chi co CPU: fallback ve `pipeline`.

Nen chon Colab runtime co GPU de parse nhanh hon. Truoc khi chay, sua `REPO_URL` o cell cau hinh ben duoi cho dung repo cua ban.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/<your-account>/<your-repo>.git"
BRANCH = "main"
PROJECT_DIR_NAME = "MinerU"
WORKSPACE = Path("/content/workspace")
PROJECT_DIR = WORKSPACE / PROJECT_DIR_NAME
PORT = 8386

# Neu chi muon client nhe de goi remote API, co the doi thanh "mineru".
# Neu chi can local pipeline CPU/GPU, co the doi thanh "mineru[core]".
MINERU_INSTALL_SPEC = "mineru[all]"
PDF_WORD_BACKEND = "auto"
MINERU_MODEL_SOURCE = "huggingface"
MINERU_VL_MODEL_NAME = "opendatalab/MinerU2.5-Pro-2604-1.2B"
MINERU_API_URL = ""
PDF_WORD_MAX_UPLOAD_MB = 128
PDF_WORD_KEEP_ARTIFACTS = True
MINERU_TIMEOUT_SECONDS = 3600
HF_TOKEN = ""

DRIVE_SYNC_ENABLED = True
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/mineru_pdf_word_backup")

STATUS_IGNORE_PREFIXES = (
    "webapp/runtime/jobs/",
    "webapp_colab.log",
    "webapp_colab.pid",
    "cloudflared.log",
    "cloudflared.pid",
)

print({
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "project_dir": str(PROJECT_DIR),
    "port": PORT,
    "python": sys.version.split()[0],
    "mineru_install_spec": MINERU_INSTALL_SPEC,
    "pdf_word_backend": PDF_WORD_BACKEND,
    "mineru_model_source": MINERU_MODEL_SOURCE,
    "mineru_vl_model_name": MINERU_VL_MODEL_NAME,
    "mineru_api_url": MINERU_API_URL or "(local auto)",
    "drive_sync_enabled": DRIVE_SYNC_ENABLED,
    "drive_backup_dir": str(DRIVE_BACKUP_DIR),
})


In [ ]:
import subprocess

WORKSPACE.mkdir(parents=True, exist_ok=True)

if not REPO_URL.strip() or "<your-" in REPO_URL:
    raise ValueError("Hay sua REPO_URL o cell cau hinh truoc khi clone tren Colab.")

def _significant_dirty_lines(lines):
    significant = []
    for raw in lines:
        line = raw.strip()
        if not line:
            continue
        path = line[3:] if len(line) > 3 else line
        normalized = path.replace('\\', '/').strip()
        if any(normalized.startswith(prefix) for prefix in STATUS_IGNORE_PREFIXES):
            continue
        significant.append(line)
    return significant

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    remote_result = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
    )
    if remote_result.returncode != 0:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "remote", "add", "origin", REPO_URL], check=True)
    else:
        current_remote = remote_result.stdout.strip()
        if current_remote and current_remote != REPO_URL:
            print(f"Cap nhat origin tu {current_remote} -> {REPO_URL}")
            subprocess.run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], check=True)

    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)

    status_result = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    )
    dirty_lines = _significant_dirty_lines(status_result.stdout.splitlines())
    if dirty_lines:
        print("Repo dang co thay doi local quan trong, bo qua git pull de tranh xung dot.")
        print("\n".join(dirty_lines[:40]))
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--short"], check=True)
print(f"Project ready at: {PROJECT_DIR}")


## Mount Google Drive & Auto Backup

Cell nay mount Google Drive de backup `webapp/runtime/jobs` giua cac session. Thu muc nay gom file PDF upload, DOCX output, Markdown, JSON va log cua MinerU.

Dat `DRIVE_SYNC_ENABLED = False` neu khong muon dung Drive.


In [ ]:
import shutil

LOCAL_JOBS_DIR = PROJECT_DIR / "webapp" / "runtime" / "jobs"
DRIVE_JOBS_DIR = DRIVE_BACKUP_DIR / "jobs"
_DRIVE_MOUNTED = False

def mount_google_drive():
    global _DRIVE_MOUNTED
    if not DRIVE_SYNC_ENABLED:
        print("Drive sync is disabled. Set DRIVE_SYNC_ENABLED = True to enable.")
        return False
    if _DRIVE_MOUNTED and Path("/content/drive/MyDrive").exists():
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
        _DRIVE_MOUNTED = True
        print(f"Google Drive mounted. Backup dir: {DRIVE_BACKUP_DIR}")
        return True
    except Exception as exc:
        print(f"Khong mount duoc Google Drive: {exc}")
        print("Session se chay binh thuong nhung khong restore/sync du lieu len Drive.")
        return False

def _sync_dir_to(src: Path, dst: Path, *, restore: bool = False):
    if restore:
        if not dst.exists():
            return 0
        src.mkdir(parents=True, exist_ok=True)
        count = 0
        for path in dst.rglob("*"):
            if not path.is_file():
                continue
            rel = path.relative_to(dst)
            target = src / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            should_copy = (not target.exists()) or (path.stat().st_mtime > target.stat().st_mtime)
            if should_copy:
                shutil.copy2(path, target)
                count += 1
        return count

    if not src.exists():
        return 0
    dst.mkdir(parents=True, exist_ok=True)
    count = 0
    for path in src.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(src)
        target = dst / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists() or path.stat().st_mtime > target.stat().st_mtime:
            shutil.copy2(path, target)
            count += 1
    return count

def restore_from_drive():
    if not DRIVE_SYNC_ENABLED:
        print("Drive sync disabled, bo qua restore.")
        return 0
    if not mount_google_drive():
        return 0
    restored = _sync_dir_to(LOCAL_JOBS_DIR, DRIVE_JOBS_DIR, restore=True)
    print(f"Da restore {restored} file tu Drive ve {LOCAL_JOBS_DIR}")
    return restored

def sync_to_drive():
    if not DRIVE_SYNC_ENABLED:
        print("Drive sync disabled, bo qua sync.")
        return 0
    if not mount_google_drive():
        return 0
    synced = _sync_dir_to(LOCAL_JOBS_DIR, DRIVE_JOBS_DIR, restore=False)
    print(f"Da sync {synced} file tu {LOCAL_JOBS_DIR} len {DRIVE_JOBS_DIR}")
    return synced

restore_from_drive()


In [ ]:
import shutil
import subprocess

subprocess.run(["apt-get", "update", "-y"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "ffmpeg", "libgl1", "libglib2.0-0", "libmagic1", "poppler-utils", "psmisc"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system", "-U", MINERU_INSTALL_SPEC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

MINERU_CLI = shutil.which("mineru")
if not MINERU_CLI:
    raise RuntimeError("Khong tim thay entrypoint `mineru` tren PATH sau khi cai dat.")

verify_cmd = [MINERU_CLI, "--help"]
verify = subprocess.run(
    verify_cmd,
    check=False,
    capture_output=True,
    text=True,
)
if verify.returncode != 0:
    detail = (verify.stderr or verify.stdout or "").strip()
    raise RuntimeError(f"Khong verify duoc MinerU sau khi cai dat.\n\n{detail[-4000:]}")

print({
    "has_gpu": has_gpu,
    "mineru_install_spec": MINERU_INSTALL_SPEC,
    "mineru_cli": MINERU_CLI,
    "verify_cmd": verify_cmd,
})


In [ ]:
import os

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV["MINERU_COMMAND"] = MINERU_CLI
RUNTIME_ENV["MINERU_PYTHON_EXE"] = sys.executable
RUNTIME_ENV["MINERU_MODEL_SOURCE"] = MINERU_MODEL_SOURCE
RUNTIME_ENV["MINERU_VL_MODEL_NAME"] = MINERU_VL_MODEL_NAME
RUNTIME_ENV["PDF_WORD_BACKEND"] = PDF_WORD_BACKEND
RUNTIME_ENV["PDF_WORD_MAX_UPLOAD_MB"] = str(int(PDF_WORD_MAX_UPLOAD_MB))
RUNTIME_ENV["PDF_WORD_KEEP_ARTIFACTS"] = "1" if PDF_WORD_KEEP_ARTIFACTS else "0"
RUNTIME_ENV["MINERU_TIMEOUT_SECONDS"] = str(int(MINERU_TIMEOUT_SECONDS))
RUNTIME_ENV["PDF_WORD_WEBAPP_PORT"] = str(PORT)

if MINERU_API_URL.strip():
    RUNTIME_ENV["MINERU_API_URL"] = MINERU_API_URL.strip()
else:
    RUNTIME_ENV.pop("MINERU_API_URL", None)

if HF_TOKEN.strip():
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Hugging Face token configured.")
else:
    print("HF_TOKEN is empty. Public downloads only.")

keys = [
    "MINERU_COMMAND",
    "MINERU_PYTHON_EXE",
    "MINERU_MODEL_SOURCE",
    "MINERU_VL_MODEL_NAME",
    "PDF_WORD_BACKEND",
    "PDF_WORD_MAX_UPLOAD_MB",
    "PDF_WORD_KEEP_ARTIFACTS",
    "MINERU_TIMEOUT_SECONDS",
    "PDF_WORD_WEBAPP_PORT",
]
if MINERU_API_URL.strip():
    keys.append("MINERU_API_URL")
print({key: RUNTIME_ENV[key] for key in keys})


In [ ]:
import json
import os
import signal
import subprocess
import time
import urllib.request
from pathlib import Path
from IPython.display import clear_output

APP_LOG = PROJECT_DIR / "webapp_colab.log"
APP_PID = PROJECT_DIR / "webapp_colab.pid"
TUNNEL_LOG = PROJECT_DIR / "cloudflared.log"
TUNNEL_PID = PROJECT_DIR / "cloudflared.pid"

def _pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def _process_status(pid_file: Path, label: str) -> str:
    if not pid_file.exists():
        return f"{label}: chua co PID"
    try:
        pid = int(pid_file.read_text().strip())
    except Exception:
        return f"{label}: PID file khong hop le"
    state = "dang chay" if _pid_alive(pid) else "da thoat"
    return f"{label}: PID {pid} {state}"

def _read_log_tail(path: Path, *, lines: int = 120, max_chars: int = 12000) -> str:
    if not path.exists():
        return "(log file chua ton tai)"
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if not raw.strip():
        return "(log file dang rong)"
    parts = raw.splitlines()
    clipped = "\n".join(parts[-max(1, int(lines)):])
    return clipped[-max_chars:]

def show_backend_logs(lines: int = 120, include_cloudflare: bool = False, max_chars: int = 12000) -> None:
    print("=== Trang thai process ===")
    print(_process_status(APP_PID, "Flask"))
    if include_cloudflare:
        print(_process_status(TUNNEL_PID, "Cloudflare"))
    print("\n=== Flask log ===")
    print(_read_log_tail(APP_LOG, lines=lines, max_chars=max_chars))
    if include_cloudflare:
        print("\n=== Cloudflare log ===")
        print(_read_log_tail(TUNNEL_LOG, lines=lines, max_chars=max_chars))

def watch_backend_logs(seconds=None, interval: int = 2, lines: int = 120, include_cloudflare: bool = False, max_chars: int = 12000) -> None:
    end_at = None if seconds is None else time.time() + max(1, int(seconds))
    tick = max(1, int(interval))
    try:
        while True:
            clear_output(wait=True)
            if end_at is None:
                print("Dang theo doi backend log lien tuc. Interrupt cell de dung.")
            else:
                remaining = max(0, int(end_at - time.time()))
                print(f"Dang theo doi backend log... con {remaining}s. Interrupt cell de dung som.")
            print()
            show_backend_logs(lines=lines, include_cloudflare=include_cloudflare, max_chars=max_chars)
            if end_at is not None and time.time() >= end_at:
                break
            time.sleep(tick)
    except KeyboardInterrupt:
        print("Da dung theo doi backend log.")

for pid_file in [APP_PID]:
    if not pid_file.exists():
        continue
    try:
        old_pid = int(pid_file.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(2)
        if _pid_alive(old_pid):
            os.kill(old_pid, signal.SIGKILL)
    except Exception:
        pass

subprocess.run(["bash", "-lc", f"fuser -k {PORT}/tcp || true"], check=False)
RUNTIME_ENV["PYTHONUNBUFFERED"] = "1"
launch_cmd = [
    sys.executable,
    "-u",
    "-c",
    (
        "from webapp.app import app; "
        f"app.run(host='0.0.0.0', port={PORT}, debug=False, use_reloader=False)"
    ),
]

log_file = APP_LOG.open("w", encoding="utf-8")
app_proc = subprocess.Popen(
    launch_cmd,
    cwd=str(PROJECT_DIR),
    env=RUNTIME_ENV,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
log_file.close()

APP_PID.write_text(str(app_proc.pid), encoding="utf-8")
status_url = f"http://127.0.0.1:{PORT}/api/status"
status_payload = None

for _ in range(90):
    if app_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(status_url, timeout=5) as resp:
            if resp.status == 200:
                status_payload = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        pass
    time.sleep(1)

log_tail = _read_log_tail(APP_LOG, lines=180, max_chars=12000)
if status_payload is None:
    raise RuntimeError(
        f"Flask khong len duoc hoac da thoat som. Return code: {app_proc.poll()}\n\n{log_tail}"
    )

readiness = status_payload.get("readiness") or {}
print(f"Flask PID: {app_proc.pid}")
print(f"Status URL OK: {status_url}")
print(json.dumps(status_payload, ensure_ascii=False, indent=2))
print()
print("Da define show_backend_logs() va watch_backend_logs() de xem log backend ngay trong notebook.")
print("Goi show_backend_logs() de xem nhanh, hoac watch_backend_logs(interval=2, include_cloudflare=True) de theo doi lien tuc.")
print()
show_backend_logs(lines=120)

if not readiness.get("ready", False):
    raise RuntimeError(
        "Webapp da len nhung MinerU chua san sang. Kiem tra payload /api/status va log ben tren."
    )


In [ ]:
from pathlib import Path
import stat
import subprocess
import urllib.request

CLOUDFLARED_BIN = Path("/usr/local/bin/cloudflared")
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not CLOUDFLARED_BIN.exists():
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)

subprocess.run([str(CLOUDFLARED_BIN), "--version"], check=True)
print(f"cloudflared ready at {CLOUDFLARED_BIN}")


In [ ]:
import os
import re
import signal
import subprocess
import time
import urllib.request
from IPython.display import HTML, display

APP_PID = PROJECT_DIR / "webapp_colab.pid"
TUNNEL_LOG = PROJECT_DIR / "cloudflared.log"
TUNNEL_PID = PROJECT_DIR / "cloudflared.pid"

if not APP_PID.exists():
    raise RuntimeError("Chua co Flask PID. Hay chay cell start Flask truoc.")

app_pid = int(APP_PID.read_text().strip())
try:
    os.kill(app_pid, 0)
except Exception as exc:
    raise RuntimeError(f"Flask PID {app_pid} khong con song. Mo log webapp_colab.log de xem loi goc.") from exc

if TUNNEL_PID.exists():
    try:
        old_tunnel_pid = int(TUNNEL_PID.read_text().strip())
        os.kill(old_tunnel_pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(old_tunnel_pid, 0)
            os.kill(old_tunnel_pid, signal.SIGKILL)
        except Exception:
            pass
    except Exception:
        pass

log_file = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    cwd=str(PROJECT_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    start_new_session=True,
)
log_file.close()

TUNNEL_PID.write_text(str(tunnel_proc.pid), encoding="utf-8")
pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
public_url = None

for _ in range(90):
    if tunnel_proc.poll() is not None:
        break
    time.sleep(1)
    log_text = TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore") if TUNNEL_LOG.exists() else ""
    match = pattern.search(log_text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError(
        "Khong tim thay trycloudflare URL. Kiem tra cloudflared.log.\n\n"
        + TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")[-4000:]
    )

for _ in range(15):
    try:
        with urllib.request.urlopen(public_url, timeout=10) as resp:
            if resp.status < 500:
                break
    except Exception:
        time.sleep(1)

print("Public URL:", public_url)
display(HTML(f'<a href="{public_url}" target="_blank">Open PDF to Word webapp</a>'))


In [ ]:
# Xem log backend/cloudflare lien tuc. Interrupt cell khi muon dung.
watch_backend_logs(interval=2, lines=140, include_cloudflare=True)


In [ ]:
import os
import signal
import time

def stop_services():
    try:
        if DRIVE_SYNC_ENABLED:
            print("Syncing converted artifacts to Google Drive before stopping...")
            sync_to_drive()
    except Exception as exc:
        print(f"Drive sync failed: {exc}")

    for pid_file in [PROJECT_DIR / "webapp_colab.pid", PROJECT_DIR / "cloudflared.pid"]:
        if not pid_file.exists():
            continue
        try:
            pid = int(pid_file.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
            try:
                os.kill(pid, 0)
                os.kill(pid, signal.SIGKILL)
            except Exception:
                pass
            pid_file.unlink(missing_ok=True)
            print(f"Stopped PID from {pid_file.name}")
        except Exception as exc:
            print(f"Could not stop {pid_file.name}: {exc}")

print("Da define stop_services() va sync_to_drive().")
print("Goi sync_to_drive() bat ky luc nao de backup artifact len Drive.")
print("Goi stop_services() de sync + tat Flask va Cloudflare tunnel.")
